# 03 — Cell-type stratified consistency + severity scan (Phase 3 prep)

Pre-Harmony diagnostic on the 4 clean SARS-CoV-2 PBMC studies
(lee_2020, wilk_2020, arunachalam_2020, schulte_schrepping_2020).
guo_2020 and mgh_acute_covid are excluded via `configs/datasets.yaml`.

Two questions:
1. **Is the cross-study near-zero correlation a cell-type-composition
   artifact, or a deeper protocol issue?** Answered by recomputing the
   cross-study response-vector Pearson r matrix *per major cell type*.
   - If per-cell-type r ≫ bulk r → composition drift → Harmony with
     cell_type as covariate.
   - If per-cell-type r still near-zero vs schulte_schrepping →
     protocol/cohort issue → drop or down-weight that study.
2. **Severity distribution per study.** If schulte_schrepping is
   systematically more severe than the others, part of the cross-study
   r heterogeneity is real biology not batch — stratify in benchmark
   splits rather than correct away.

Pure laptop CPU.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.insert(0, str(Path("..") / "src"))
from trinetravir.data.download import load_dataset_config

REPO = Path("..").resolve()
RAW_DIR = REPO / "data" / "raw"
CFG = REPO / "configs" / "datasets.yaml"

_, studies = load_dataset_config(CFG)
clean_ids = [sid for sid, s in studies.items() if not s.excluded and s.source == "cellxgene"]
print("Clean (non-excluded) studies:")
for sid in clean_ids:
    print(f"  - {sid}")
print()
for sid, s in studies.items():
    if s.excluded:
        first_line = s.exclusion_reason.splitlines()[0] if s.exclusion_reason else ""
        print(f"EXCLUDED  {sid}: {first_line}")

## Coarse cell-type binning
Map heterogeneous Cell Ontology strings to 5 coarse buckets. Conservative — anything ambiguous is dropped from the stratified analysis.

In [ ]:
def coarse_cell_type(ct: str) -> str | None:
    s = ct.lower()
    # Order matters: monocyte/macrophage tokens checked before generic 'cd' tokens
    if "monocyte" in s or "macrophage" in s:
        return "monocyte"
    if "natural killer" in s or s.startswith("nk") or " nk " in s:
        return "NK"
    if (
        "plasmablast" in s
        or "plasma cell" in s
        or s.endswith(" b cell")
        or s == "b cell"
        or "memory b cell" in s
        or ("b cell" in s and "t cell" not in s)
    ):
        return "B"
    if "cd4" in s and "t cell" in s:
        return "CD4T"
    if "cd8" in s and "t cell" in s:
        return "CD8T"
    return None  # exclude platelets, dendritic, progenitors, undifferentiated 't cell', etc.


loaded = {}
for sid in clean_ids:
    p = RAW_DIR / f"{sid}.h5ad"
    if not p.exists():
        print(f"  missing on disk: {p.name}; skipping")
        continue
    a = ad.read_h5ad(p)
    a.obs["coarse"] = a.obs["cell_type"].astype(str).map(coarse_cell_type).astype("category")
    loaded[sid] = a
    n_keep = a.obs["coarse"].notna().sum()
    n_total = a.n_obs
    print(
        f"{sid:<28}  n_total={n_total:>6}  n_in_5_buckets={n_keep:>6}  ({n_keep / n_total * 100:.1f}%)"
    )

In [ ]:
# Per-study × per-bucket cell counts, split by donor_disease_status
rows = []
for sid, a in loaded.items():
    for bucket in ["monocyte", "CD4T", "CD8T", "B", "NK"]:
        m = a.obs["coarse"] == bucket
        n_d = int(
            ((a.obs["coarse"] == bucket) & (a.obs["donor_disease_status"] == "diseased")).sum()
        )
        n_h = int(
            ((a.obs["coarse"] == bucket) & (a.obs["donor_disease_status"] == "healthy")).sum()
        )
        rows.append({"study_id": sid, "bucket": bucket, "n_diseased": n_d, "n_healthy": n_h})
counts = pd.DataFrame(rows).pivot_table(
    index="study_id", columns="bucket", values=["n_diseased", "n_healthy"], fill_value=0
)
pd.set_option("display.width", 220)
counts

## Cross-study Pearson r per cell-type bucket
Same response-vector definition as notebook 02 (mean(diseased) − mean(healthy) on log1p-normalized counts), but restricted to a single cell-type bucket. MIN_PER_GROUP guards against bucket-study combinations w/ too few cells.

In [ ]:
MIN_PER_GROUP = 50

# Pre-normalize each study once (saves repeated work across buckets)
normed = {}
for sid, a in loaded.items():
    a2 = a.copy()
    sc.pp.normalize_total(a2, target_sum=1e4)
    sc.pp.log1p(a2)
    normed[sid] = a2
print("Normalized.")


def response(adata_norm, bucket):
    sars = adata_norm.obs["virus"].isin(["sars_cov_2", "mock"])
    in_bucket = adata_norm.obs["coarse"] == bucket
    sub = adata_norm[sars & in_bucket]
    n_d = int((sub.obs["donor_disease_status"] == "diseased").sum())
    n_h = int((sub.obs["donor_disease_status"] == "healthy").sum())
    if n_d < MIN_PER_GROUP or n_h < MIN_PER_GROUP:
        return None
    d = sub[sub.obs["donor_disease_status"] == "diseased"].X
    h = sub[sub.obs["donor_disease_status"] == "healthy"].X
    return pd.Series(
        np.asarray(d.mean(axis=0)).ravel() - np.asarray(h.mean(axis=0)).ravel(),
        index=adata_norm.var_names,
    )


per_bucket_corrs = {}
for bucket in ["monocyte", "CD4T", "CD8T", "B", "NK"]:
    rvs = {}
    for sid, a in normed.items():
        rv = response(a, bucket)
        if rv is not None:
            rvs[sid] = rv
    if len(rvs) < 2:
        print(f"\n=== {bucket}: not enough studies w/ >= {MIN_PER_GROUP} cells per group")
        continue
    studies_in = list(rvs.keys())
    shared = rvs[studies_in[0]].index
    for s in studies_in[1:]:
        shared = shared.intersection(rvs[s].index)
    aligned = pd.DataFrame({s: rvs[s].loc[shared] for s in studies_in})
    corr = aligned.corr(method="pearson")
    per_bucket_corrs[bucket] = corr
    off = corr.values[~np.eye(len(corr), dtype=bool)]
    print(f"\n=== {bucket} ({len(studies_in)} studies, {len(shared)} shared genes) ===")
    print(corr.round(3))
    print(f"mean off-diag r = {off.mean():.3f}  (range [{off.min():.3f}, {off.max():.3f}])")

## Per-study summary: per-bucket vs bulk r
How much does stratification rescue each study's mean r? Big lift → composition drift fixable. Small/none → deeper issue.

In [ ]:
if per_bucket_corrs:
    summary_rows = []
    all_studies = sorted({s for c in per_bucket_corrs.values() for s in c.index})
    for sid in all_studies:
        row = {"study_id": sid}
        for bucket, corr in per_bucket_corrs.items():
            if sid not in corr.index:
                row[bucket] = np.nan
                continue
            others = [c for c in corr.columns if c != sid]
            row[bucket] = corr.loc[sid, others].mean()
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows).set_index("study_id")
    summary["mean_per_bucket"] = summary.mean(axis=1, skipna=True)
    print("Per-study mean cross-study r WITHIN each bucket (vs other studies):")
    print(summary.round(3))

## Severity distribution scan
Look for severity-like obs columns. cellxgene Census strips most study-specific metadata, so this is largely a survey of what's available; gaps go on the documentation TODO list.

In [ ]:
severity_keywords = [
    "severity",
    "who",
    "severe",
    "mild",
    "moderate",
    "critical",
    "icu",
    "ventil",
    "oxygen",
]
for sid, a in loaded.items():
    cols = list(a.obs.columns)
    matches = [c for c in cols if any(k in c.lower() for k in severity_keywords)]
    print(f"\n=== {sid} ===")
    if matches:
        for c in matches:
            print(f"  obs.{c}: top values =", a.obs[c].value_counts().head(5).to_dict())
    else:
        print("  NO severity-like obs column found in cellxgene Census record")
        # Fall back: print development_stage and disease_ontology_term_id distribution as proxies
        for proxy in ["development_stage", "disease_ontology_term_id"]:
            if proxy in cols:
                vc = a.obs[proxy].value_counts().head(5)
                print(f"  proxy obs.{proxy} top values:", vc.to_dict())

## Verdict
Compare bulk vs per-bucket r. If schulte_schrepping mean per-bucket r climbs > 0.3 → composition drift, harmonization with cell_type covariate is the fix. If it stays near zero → deeper issue, drop or down-weight.